# RoPE — Rotary Position Embedding — Exercise

Companion to [Positional Encoding Part 2 § RoPE](https://tanulsingh.github.io/blog/positional-encoding-relative#rope-rotary-position-embedding-su-et-al-2021).

**This is the most important and most subtle method.** RoPE powers Llama, Mistral, Qwen, Gemma, and virtually every major LLM since 2023.

The key insight you'll prove numerically: when Q and K are rotated by their absolute positions, the attention score depends only on the *relative* position $j - i$.

## Setup

In [ ]:
# TODO: imports — torch, math

## TODO 1 — RoPE frequencies

Same frequency formula as sinusoidal PE: $\theta_i = 1 / 10000^{2i/d_{\text{head}}}$ for $i = 0, \ldots, d_{\text{head}}/2 - 1$.

Note: this is per-attention-head dimension `d_head`, not `d_model`.

In [ ]:
def get_rope_frequencies(d_head: int, base: float = 10000.0) -> 'torch.Tensor':
    """Return shape (d_head // 2,) tensor of frequencies theta_i."""
    # TODO 1: implement
    pass

## TODO 2 — Rotate a single 4D vector

Before tackling batched Q and K, rotate one 4D vector to make sure the mechanics work.

For `q = [q_0, q_1, q_2, q_3]` at position `pos`, with frequencies `[θ_0, θ_1]`:
- Pair $(q_0, q_1)$ gets rotated by angle $\theta_0 \cdot pos$
- Pair $(q_2, q_3)$ gets rotated by angle $\theta_1 \cdot pos$

Use the standard 2D rotation:
$$\begin{bmatrix} q'_{2i} \\ q'_{2i+1} \end{bmatrix} = \begin{bmatrix} \cos\phi & -\sin\phi \\ \sin\phi & \cos\phi \end{bmatrix} \begin{bmatrix} q_{2i} \\ q_{2i+1} \end{bmatrix}$$

In [ ]:
# TODO 2:
#   - Pick q = torch.tensor([1.0, 0.5, 0.8, -0.3]), pos = 1, d_head = 4
#   - Compute the angles for each pair
#   - Apply the 2x2 rotation to each pair
#   - Compare with the worked example in the blog (cat at pos 1)

## TODO 3 — Batched RoPE on Q (or K)

Now generalize. Given:
- `x`: shape `(batch, seq_len, n_heads, d_head)`
- `positions`: shape `(seq_len,)` of integer positions

Return `x` with each `(2i, 2i+1)` dimension pair rotated by $\theta_i \cdot pos$.

**Tip:** instead of building rotation matrices, use the equivalent formulation:
$$x'_{2i} = x_{2i} \cos\phi - x_{2i+1} \sin\phi$$
$$x'_{2i+1} = x_{2i} \sin\phi + x_{2i+1} \cos\phi$$

This avoids materializing block-diagonal matrices and is what real RoPE implementations do.

In [ ]:
def apply_rope(x: 'torch.Tensor', positions: 'torch.Tensor') -> 'torch.Tensor':
    """Apply RoPE to a Q or K tensor.

    Args:
        x: (batch, seq_len, n_heads, d_head)
        positions: (seq_len,) integer positions

    Returns:
        Same shape as x, with each dim pair rotated.
    """
    # TODO 3: implement
    pass

## TODO 4 — The big experiment: dot product depends only on relative distance

This is the punchline of RoPE. Demonstrate numerically:

Let $q$, $k$ be random vectors of shape `(d_head,)`. Compute:
- $q' = $ rotate $q$ by position $i$
- $k' = $ rotate $k$ by position $j$
- $\text{score}(i, j) = q' \cdot k'$

Claim: $\text{score}(3, 7) \approx \text{score}(100, 104) \approx \text{score}(1000, 1004)$. Same offset $\Rightarrow$ same dot product.

In [ ]:
# TODO 4:
#   - Pick a random q and k of shape (d_head,)
#   - For (i, j) in [(3, 7), (100, 104), (1000, 1004), (5000, 5004)]:
#       compute score(i, j)
#   - Assert they're all approximately equal (atol=1e-4)
#   - Then verify that score(3, 7) != score(3, 9) (different offset, different score)